# Big Data Out-of-Core & Microservicio MLOps con FastAPI
### Detección de Fraude Transaccional Masivo (PaySim1) & Docker

**Autor:** Luis Monge  
**Metodología:** CRISP-DM  
**Dataset Oficial:** PaySim1 Synthetic Financial Logs (Kaggle)

## 1. Carga Adaptativa y Procesamiento Distribuido Dask

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from fastapi.testclient import TestClient

from src.data_loader import load_dataset
from src.bigdata_processor import ProcesadorTransaccionesDistribuidas
from src.nosql_connector import ConectorNoSQLHibrido
from src.api import app

df_txns = load_dataset(force_sample=False)
if isinstance(df_txns, list):
    df_txns = pd.DataFrame(df_txns)
print(f"Transacciones cargadas: {len(df_txns)}")
print(df_txns.head(3))

## 2. Ejecución del Grafo Perezoso (Lazy Evaluation en TRANSFER y CASH_OUT)

In [2]:
procesador = ProcesadorTransaccionesDistribuidas(n_particiones=4)
resumen_dask = procesador.procesar_grafo_dask(df_txns)
print("Resumen de Fraude y Discrepancias (Post-.compute()):")
print(resumen_dask)

## 3. Persistencia NoSQL en MongoDB y Respaldo en Data Lake

In [3]:
conector = ConectorNoSQLHibrido()
insertados = conector.persistir_reportes_agregados(resumen_dask)
manifiesto = conector.respaldar_en_data_lake(resumen_dask)

print(f"Documentos JSON indexados en MongoDB: {insertados}")
print(f"Particiones Parquet escritas en Cloud Data Lake: {manifiesto['particiones_escritas']}")

## 4. Prueba del Microservicio REST con FastAPI TestClient

In [4]:
cliente = TestClient(app)
salud = cliente.get('/health')
print(f"Endpoint /health [{salud.status_code}]: {salud.json()}")

payload = {
    'transaccion_id': 'TXN-LIVE-001',
    'tipo_operacion': 'TRANSFER',
    'amount': 350000.0,
    'oldbalanceOrg': 350000.0,
    'newbalanceOrig': 0.0,
    'oldbalanceDest': 0.0,
    'newbalanceDest': 350000.0
}
scoring = cliente.post('/predict', json=payload)
print(f"Endpoint /predict [{scoring.status_code}]: {scoring.json()}")